# Lab 5: Data Visualization Portfolio

**Course:** Data Analytics Laboratory  
**Contributors:** Abdul Ahad, Abdullah Jabbar  

---

## Objectives
- Line plots for time series data
- Bar charts and histograms for distributions
- Scatter plots for correlation analysis
- Box plots and violin plots for statistical distributions
- Heatmaps for correlation matrices
- Customize plots: colors, labels, titles, legends
- Create subplots and multi-panel figures

**Dataset:** COVID-19 Statistics (Simulated)  
**Target:** 10+ different chart types

## Setup & Data Generation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set global style
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})
sns.set_theme(style="whitegrid", palette="husl")

# Generate COVID-19-like dataset
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')

countries = {
    'Pakistan': {'pop': 231e6, 'peak': 5000, 'peak_day': 60},
    'India': {'pop': 1408e6, 'peak': 40000, 'peak_day': 75},
    'UK': {'pop': 67e6, 'peak': 15000, 'peak_day': 45},
    'USA': {'pop': 334e6, 'peak': 80000, 'peak_day': 50},
    'Germany': {'pop': 84e6, 'peak': 20000, 'peak_day': 55},
}

covid_data = []
for country, params in countries.items():
    for i, date in enumerate(dates):
        # Bell curve for daily cases
        cases = int(params['peak'] * np.exp(-0.5 * ((i - params['peak_day']) / 30) ** 2)
                    * (1 + 0.3 * np.random.randn()))
        cases = max(0, cases)
        deaths = int(cases * np.random.uniform(0.01, 0.03))
        recovered = int(cases * np.random.uniform(0.85, 0.95))
        vaccinated = min(params['pop'] * 0.8, params['pop'] * 0.8 * (1 / (1 + np.exp(-0.03 * (i - 120)))))
        covid_data.append({
            'Date': date, 'Country': country,
            'Daily_Cases': cases, 'Daily_Deaths': deaths,
            'Recovered': recovered, 'Population': params['pop'],
            'Vaccinated': int(vaccinated)
        })

df = pd.DataFrame(covid_data)
df['Cumulative_Cases'] = df.groupby('Country')['Daily_Cases'].cumsum()
df['Vaccination_Rate'] = (df['Vaccinated'] / df['Population'] * 100).round(2)

print(f"Dataset shape: {df.shape}")
print(f"Countries: {df['Country'].unique()}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
df.head()

## Chart 1: Line Plot — Daily Cases Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for country in df['Country'].unique():
    country_data = df[df['Country'] == country]
    # 7-day rolling average for smoother lines
    rolling = country_data['Daily_Cases'].rolling(7).mean()
    ax.plot(country_data['Date'], rolling, linewidth=2, label=country)

ax.set_title('COVID-19 Daily Cases (7-Day Rolling Average)', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Cases')
ax.legend(loc='upper right', framealpha=0.9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart_01_line.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 1: Line Plot ✓")

## Chart 2: Bar Chart — Total Cases by Country

In [ ]:
total_cases = df.groupby('Country')['Daily_Cases'].sum().sort_values(ascending=True)
colors = sns.color_palette("viridis", len(total_cases))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(total_cases.index, total_cases.values, color=colors, edgecolor='white', height=0.6)

for bar, val in zip(bars, total_cases.values):
    ax.text(bar.get_width() + total_cases.max() * 0.01,
            bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=11, fontweight='bold')

ax.set_title('Total COVID-19 Cases by Country', fontweight='bold')
ax.set_xlabel('Total Cases')
ax.set_xlim(0, total_cases.max() * 1.15)
plt.tight_layout()
plt.savefig('chart_02_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 2: Bar Chart ✓")

## Chart 3: Histogram — Distribution of Daily Cases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
pak_cases = df[df['Country'] == 'Pakistan']['Daily_Cases']
axes[0].hist(pak_cases, bins=40, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].axvline(pak_cases.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {pak_cases.mean():.0f}')
axes[0].axvline(pak_cases.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {pak_cases.median():.0f}')
axes[0].set_title('Pakistan — Daily Cases Distribution', fontweight='bold')
axes[0].set_xlabel('Daily Cases')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# KDE overlay
for country in df['Country'].unique():
    cases = df[df['Country'] == country]['Daily_Cases']
    cases = cases[cases > 100]  # Filter noise
    if len(cases) > 10:
        axes[1].hist(cases, bins=30, alpha=0.3, label=country, density=True)

axes[1].set_title('Daily Cases Distribution (All Countries)', fontweight='bold')
axes[1].set_xlabel('Daily Cases')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig('chart_03_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 3: Histogram ✓")

## Chart 4: Scatter Plot — Cases vs Deaths Correlation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for country in df['Country'].unique():
    cdata = df[df['Country'] == country]
    ax.scatter(cdata['Daily_Cases'], cdata['Daily_Deaths'],
              alpha=0.4, s=20, label=country)

# Trend line (all data)
from numpy.polynomial import polynomial as P
mask = df['Daily_Cases'] > 0
x = df.loc[mask, 'Daily_Cases'].values
y = df.loc[mask, 'Daily_Deaths'].values
coeffs = np.polyfit(x, y, 1)
x_line = np.linspace(0, x.max(), 100)
ax.plot(x_line, np.polyval(coeffs, x_line), 'r--', linewidth=2, label='Trend Line')

ax.set_title('Daily Cases vs Deaths — Correlation Analysis', fontweight='bold')
ax.set_xlabel('Daily Cases')
ax.set_ylabel('Daily Deaths')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('chart_04_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation coefficient
corr = np.corrcoef(x, y)[0, 1]
print(f"Chart 4: Scatter Plot ✓ (r = {corr:.4f})")

## Chart 5: Box Plot — Daily Cases Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Filter to peak months for clearer visualization
peak_data = df[(df['Date'] >= '2023-01-15') & (df['Date'] <= '2023-05-15')]

sns.boxplot(data=peak_data, x='Country', y='Daily_Cases', palette='Set2', ax=ax,
            flierprops=dict(marker='o', markersize=3, alpha=0.5))

ax.set_title('Daily Cases Distribution by Country (Peak Period)', fontweight='bold')
ax.set_xlabel('Country')
ax.set_ylabel('Daily Cases')
plt.tight_layout()
plt.savefig('chart_05_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 5: Box Plot ✓")

## Chart 6: Violin Plot — Monthly Case Distribution

In [ ]:
# Create month column
df['Month'] = df['Date'].dt.month

pak_data = df[df['Country'] == 'Pakistan'].copy()
pak_data = pak_data[pak_data['Month'].isin([1, 2, 3, 4, 5, 6])]

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=pak_data, x='Month', y='Daily_Cases', palette='muted',
               inner='box', ax=ax)

month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun'}
ax.set_xticklabels([month_names.get(int(t.get_text()), t.get_text()) for t in ax.get_xticklabels()])
ax.set_title('Pakistan — Monthly Daily Cases Distribution (Violin Plot)', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Daily Cases')
plt.tight_layout()
plt.savefig('chart_06_violin.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 6: Violin Plot ✓")

## Chart 7: Heatmap — Correlation Matrix

In [ ]:
# Prepare numeric columns for correlation
numeric_cols = df[['Daily_Cases', 'Daily_Deaths', 'Recovered', 'Vaccinated', 'Vaccination_Rate', 'Cumulative_Cases']]
corr_matrix = numeric_cols.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f',
            cmap='RdYlBu_r', center=0, square=True,
            linewidths=1, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1, ax=ax)

ax.set_title('COVID-19 Metrics — Correlation Heatmap', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('chart_07_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 7: Heatmap ✓")

## Chart 8: Area Chart — Cumulative Cases

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for country in df['Country'].unique():
    cdata = df[df['Country'] == country]
    ax.fill_between(cdata['Date'], cdata['Cumulative_Cases'], alpha=0.3)
    ax.plot(cdata['Date'], cdata['Cumulative_Cases'], linewidth=1.5, label=country)

ax.set_title('Cumulative COVID-19 Cases Over Time', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Cases')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart_08_area.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 8: Area Chart ✓")

## Chart 9: Pie Chart — Case Share by Country

In [ ]:
total = df.groupby('Country')['Daily_Cases'].sum()
colors = sns.color_palette("Set2", len(total))
explode = [0.05] * len(total)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
wedges, texts, autotexts = axes[0].pie(total, labels=total.index, autopct='%1.1f%%',
                                        colors=colors, explode=explode,
                                        startangle=140, pctdistance=0.8)
for t in autotexts:
    t.set_fontweight('bold')
axes[0].set_title('Total Cases Share by Country', fontweight='bold')

# Donut chart
wedges2, texts2, autotexts2 = axes[1].pie(total, labels=total.index, autopct='%1.1f%%',
                                            colors=colors, explode=explode,
                                            startangle=140, pctdistance=0.8,
                                            wedgeprops=dict(width=0.5))
for t in autotexts2:
    t.set_fontweight('bold')
center_circle = plt.Circle((0, 0), 0.30, fc='white')
axes[1].add_artist(center_circle)
axes[1].set_title('Total Cases Share (Donut)', fontweight='bold')

plt.tight_layout()
plt.savefig('chart_09_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 9: Pie + Donut Chart ✓")

## Chart 10: Stacked Bar — Monthly Cases by Country

In [ ]:
monthly = df.groupby([df['Date'].dt.to_period('M'), 'Country'])['Daily_Cases'].sum().unstack()
monthly.index = monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 6))
monthly.plot(kind='bar', stacked=True, ax=ax, colormap='viridis', edgecolor='white', width=0.8)

ax.set_title('Monthly COVID-19 Cases by Country (Stacked)', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Total Cases')
ax.legend(title='Country', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('chart_10_stacked.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 10: Stacked Bar ✓")

## Chart 11: Subplots — Multi-Panel Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('COVID-19 Analytics Dashboard', fontsize=16, fontweight='bold', y=1.01)

# Panel 1: Line plot — top 3 countries
for country in ['USA', 'India', 'Germany']:
    cdata = df[df['Country'] == country]
    axes[0, 0].plot(cdata['Date'], cdata['Daily_Cases'].rolling(7).mean(),
                    linewidth=2, label=country)
axes[0, 0].set_title('Daily Cases Trend (Top 3)')
axes[0, 0].legend()
axes[0, 0].set_ylabel('Cases')

# Panel 2: Bar — death rate by country
death_rate = df.groupby('Country').apply(
    lambda x: x['Daily_Deaths'].sum() / x['Daily_Cases'].sum() * 100
).sort_values()
colors = ['#2ecc71' if v < 2 else '#e74c3c' for v in death_rate.values]
axes[0, 1].barh(death_rate.index, death_rate.values, color=colors)
axes[0, 1].set_title('Case Fatality Rate (%)')
axes[0, 1].set_xlabel('Death Rate (%)')

# Panel 3: Vaccination progress
for country in df['Country'].unique():
    cdata = df[df['Country'] == country]
    axes[1, 0].plot(cdata['Date'], cdata['Vaccination_Rate'], linewidth=2, label=country)
axes[1, 0].set_title('Vaccination Progress (%)')
axes[1, 0].set_ylabel('Vaccination Rate (%)')
axes[1, 0].axhline(y=70, color='red', linestyle='--', alpha=0.7, label='Herd Immunity Target')
axes[1, 0].legend(fontsize=8)

# Panel 4: Scatter — peak cases vs population
peak_cases = df.groupby('Country')['Daily_Cases'].max()
pop = df.groupby('Country')['Population'].first() / 1e6
axes[1, 1].scatter(pop, peak_cases, s=200, c=sns.color_palette("Set2", 5),
                    edgecolors='black', zorder=5)
for country in pop.index:
    axes[1, 1].annotate(country, (pop[country], peak_cases[country]),
                        textcoords="offset points", xytext=(8, 5), fontsize=10)
axes[1, 1].set_title('Peak Cases vs Population')
axes[1, 1].set_xlabel('Population (millions)')
axes[1, 1].set_ylabel('Peak Daily Cases')

plt.tight_layout()
plt.savefig('chart_11_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 11: Multi-Panel Dashboard ✓")

## Chart 12: Pair Plot — Multivariate Analysis

In [ ]:
# Sample data for pair plot
sample = df[df['Daily_Cases'] > 100].sample(500, random_state=42)
pair_cols = ['Daily_Cases', 'Daily_Deaths', 'Recovered', 'Vaccination_Rate']

g = sns.pairplot(sample[pair_cols + ['Country']], hue='Country',
                  diag_kind='kde', plot_kws={'alpha': 0.5, 's': 15},
                  height=2.5, aspect=1)
g.figure.suptitle('COVID-19 Metrics — Pair Plot', y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_12_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 12: Pair Plot ✓")

---
## Visualization Portfolio Summary

| # | Chart Type | Description | Status |
|---|---|---|---|
| 1 | Line Plot | Daily cases time series (7-day avg) | Done |
| 2 | Bar Chart | Total cases by country | Done |
| 3 | Histogram | Daily cases distribution | Done |
| 4 | Scatter Plot | Cases vs deaths correlation | Done |
| 5 | Box Plot | Cases distribution by country | Done |
| 6 | Violin Plot | Monthly case distribution | Done |
| 7 | Heatmap | Correlation matrix | Done |
| 8 | Area Chart | Cumulative cases | Done |
| 9 | Pie + Donut | Case share by country | Done |
| 10 | Stacked Bar | Monthly cases breakdown | Done |
| 11 | Multi-Panel | Analytics dashboard (4 subplots) | Done |
| 12 | Pair Plot | Multivariate analysis | Done |

**Total: 12 chart types** (exceeds 10+ requirement)

---
*Lab 5 completed successfully.*